In [ ]:
import sys
sys.path.append('..')

from hybrid_rag.pipeline import HybridRAGPipeline

hybrid = HybridRAGPipeline()

In [ ]:
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

TEST_QUESTIONS = [
    "What was NVIDIA's total revenue for the most recent fiscal year?",
    "What percentage of NVIDIA's revenue came from data center products?",
    "What was Amazon Web Services revenue for the most recent fiscal year?",
    "What is Microsoft's stated strategy for artificial intelligence investments?",
    "What risks related to content licensing did Netflix identify?",
]

for q in TEST_QUESTIONS:
    print(f"\n{'#'*60}")
    print(f"  Q: {q}")
    print(f"{'#'*60}")

    print("Running Vector...")
    r_vec = vec.ask(q)

    print("Running Vectorless...")
    r_vl = vl.ask(q)

    print("Running Hybrid...")
    r_hybrid = hybrid.ask(q)

    print(f"\n[VECTOR]     {r_vec['answer'][:200]}")
    print(f"  ↳ time: {r_vec['total_time']}s")

    print(f"\n[BM25]       {r_vl['answer'][:200]}")
    print(f"  ↳ time: {r_vl['total_time']}s")

    print(f"\n[HYBRID]     {r_hybrid['answer'][:200]}")
    print(f"  ↳ time: {r_hybrid['total_time']}s  "
          f"(vector: {r_hybrid['vector_latency']}s  "
          f"bm25: {r_hybrid['bm25_latency']}s  "
          f"rerank: {r_hybrid['rerank_latency']}s)")
    print(f"  ↳ candidates: vector={r_hybrid['vector_candidates']} "
          f"bm25={r_hybrid['bm25_candidates']} "
          f"fused={r_hybrid['fused_candidates']} "
          f"final={len(r_hybrid['retrieved'])}")

In [ ]:
results = vec.collection.get(
    include=["metadatas"]
)

companies = set(
    m["company"]
    for m in results["metadatas"]
)

print(companies)
print(len(results["metadatas"]))

In [ ]:
q = "What risks related to content licensing did Netflix identify?"

ret = vec.ask(q)

print("Retrieved Chunks:", len(ret["retrieved"]))

for i, chunk in enumerate(ret["retrieved"]):
    print("\n" + "="*80)
    print(f"Chunk {i+1}")
    print("="*80)
    print(chunk["text"][:2000])

In [ ]:
from vector_rag.retriever     import VectorRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

ret = vec.retriever.retrieve(
    "What risks related to content licensing did Netflix identify?"
)

In [ ]:
from vector_rag.retriever import retrieve

ret = retrieve(
    "What risks related to content licensing did Netflix identify?",
    vec.collection,
    vec.parent_lookup,
)

print(ret)

In [ ]:
from collections import Counter

results = vec.collection.get(include=["metadatas"])

print(Counter(
    m["company"]
    for m in results["metadatas"]
))


In [ ]:
import json

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    data = json.load(f)

print("Parents:", len(data["parents"]))
print("Children:", len(data["children"]))

companies = {}

for c in data["children"]:
    company = c["company"]
    companies[company] = companies.get(company, 0) + 1

print(companies)

In [ ]:
import json

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    data = json.load(f)

sources = {}

for c in data["children"]:
    company = c["company"]

    if company not in sources:
        sources[company] = set()

    sources[company].add(c["source"])

for company, s in sources.items():
    print(company)
    print(list(s)[:10])
    print()

In [ ]:
results = vec.collection.get(include=["metadatas"])

sources = {}

for m in results["metadatas"]:
    company = m["company"]

    if company not in sources:
        sources[company] = set()

    sources[company].add(m["source"])

for company, s in sources.items():
    print(company)
    print(list(s))

In [ ]:
import config
print(config.CHROMA_PERSIST_DIR)